In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers":12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Deep learning is so much fun"
txt2 = "It was breakthrough in LLMs"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
    super().__init__()
    assert (d_out % num_heads == 0)

    self.d_out = d_out
    self.num_heads = num_heads
    self.head_dim = d_out // num_heads

    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.out_proj = nn.Linear(d_out, d_out)
    self.dropout = nn.Dropout(dropout)
    self.register_buffer(
        "mask",
        torch.triu(torch.ones(context_length, context_length),
                   diagonal=1)
    )
  def forward(self, x):
    b, num_tokens, d_in = x.shape

    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)

    keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
    values = values.view(b, num_tokens, self.num_heads, self.head_dim)
    queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

    keys = keys.transpose(1, 2)
    queries = queries.transpose(1, 2)
    values = values.transpose(1, 2)

    attn_scores = queries @ keys.transpose(2, 3)

    mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

    attn_scores.masked_fill_(mask_bool, -torch.inf)

    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
    attn_weights = self.dropout(attn_weights)

    context_vec = (attn_weights @ values).transpose(1, 2)

    context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
    context_vec = self.out_proj(context_vec)

    return context_vec


In [ ]:
import torch
import torch.nn as nn

class DummyGPTModel(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.token_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
    self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
    self.drop_emb = nn.Dropout(cfg["drop_rate"])

    self.trf_block = nn.Sequential(
        *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])]
    )
    self.final_norm = DummyLayerNorm(cfg["emb_dim"])
    self.out_head = nn.Linear(
        cfg["emb_dim"], cfg["vocab_size"], bias=False
    )

  def forward(self, in_idx):
    batch_size, seq_length = in_idx.shape
    token_embeds = self.token_emb(in_idx)
    pos_embeds = self.pos_emb(torch.arange(seq_length, device=in_idx.device))
    x = token_embeds + pos_embeds
    x = self.drop_emb(x)
    x = self.trf_block(x)
    x = self.final_norm(x)
    logits = self.out_head(x)

    return logits

class DummyTransformerBlock(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.att = MultiHeadAttention(
        d_in = cfg["emb_dim"],
        d_out = cfg["emb_dim"],
        context_length = cfg["context_length"],
        num_heads = cfg["n_heads"],
        dropout = cfg["drop_rate"],
        qkv_bias = cfg["qkv_bias"]
    )
    self.ff = FeedForward(cfg)
    self.norm1 = DummyLayerNorm(cfg["emb_dim"])
    self.norm2 = DummyLayerNorm(cfg["emb_dim"])
    self.dropout_shortcut = nn.Dropout(cfg["drop_rate"])

  def forward(self, x):
    shortcut = x
    x = self.norm1(x)
    x = self.att(x)
    x = self.dropout_shortcut(x)
    x += shortcut


    shortcut = x
    x = self.norm2(x)
    x = self.ff(x)
    x = self.dropout_shortcut(x)
    x += shortcut
    return x

class DummyLayerNorm(nn.Module):
  def __init__(self, emb_dim):
    super().__init__()
    self.eps = 1e-5
    self.scale = nn.Parameter(torch.ones(emb_dim))
    self.shift = nn.Parameter(torch.zeros(emb_dim))

  def forward(self, x):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    norm_x = (x - mean) / torch.sqrt(var + self.eps)
    return self.scale*norm_x + self.shift

class GELU(nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, x):
    return 0.5* x* (1 + torch.tanh(
        torch.sqrt(torch.tensor(2.0 /torch.pi))*(x+0.044715*pow(x, 3))
    ))

class FeedForward(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
        GELU(),
        nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"])
    )

  def forward(self, x):
    return self.layers(x)

In [ ]:
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)
out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

Input batch:
 tensor([[29744,  4673,   318,   523,   881,  1257],
        [ 1026,   373, 19304,   287, 27140, 10128]])

Output shape: torch.Size([2, 6, 50257])
tensor([[[ 0.3669, -0.1964, -0.0385,  ..., -0.1362, -0.2178, -0.8006],
         [ 0.5070,  0.6923,  0.2859,  ..., -0.4652, -0.3014, -0.0853],
         [ 1.5037, -0.6379,  0.1118,  ...,  0.3886, -0.1033, -0.4449],
         [ 0.0745,  0.5688,  0.5261,  ...,  0.7108, -0.1337, -0.5986],
         [ 1.0921,  0.6043, -0.0759,  ..., -0.8405, -0.5282,  0.6168],
         [-0.2072,  0.3221,  0.2437,  ..., -0.3562,  0.1211, -1.3098]],

        [[-0.0759, -0.1267, -0.3782,  ..., -0.0153,  0.8423, -0.8778],
         [-0.0772,  0.6561,  0.9376,  ...,  0.2950, -0.7439, -0.0947],
         [ 0.9560,  0.5989, -0.0369,  ...,  0.6694, -0.5107,  0.0524],
         [-0.5412,  0.6308, -0.4691,  ...,  1.0490, -0.0776, -0.5587],
         [ 1.4058,  0.0788,  0.7331,  ...,  0.0768, -0.1254, -0.8377],
         [-0.7553,  0.9974, -0.1711,  ..., -1.3821,  0.35

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"{total_params:,}")

163,009,536


In [ ]:
print(model.token_emb.weight.shape)
print(model.out_head.weight.shape)

torch.Size([50257, 768])
torch.Size([50257, 768])


In [ ]:
actual_total_params_of_gpt2 = total_params - sum(p.numel() for p in model.out_head.parameters())
print(f"{actual_total_params_of_gpt2:,}")

124,412,160


In [ ]:
total_size = total_params * 4
total_size_of_gpt2 = actual_total_params_of_gpt2 * 4
total_size_of_gpt2_mb = total_size_of_gpt2 / (1024*1024)
total_size_mb = total_size / (1024 * 1024)
print(total_size_mb, "MB")
print(total_size_of_gpt2_mb, "MB")

621.83203125 MB
474.5947265625 MB
